# 🎬 VibeMV Complete Studio - GPU Edition

**All-in-one notebook** for VibeMV video and 3D generation.

## What This Does

- ✅ Generate images from your timeline
- 🎥 Create videos with camera motion
- 🎲 Generate 3D models (optional)
- 📥 Download everything

## Setup

1. **Enable GPU**: Runtime → Change runtime type → T4 GPU
2. **Run all cells** in order
3. **Upload timeline** when prompted

**Time:** ~5-10 minutes for a complete video

In [ ]:
# @title ✅ Check GPU & Install Dependencies (2-3 minutes)
import torch

# Check GPU
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('❌ No GPU! Enable it: Runtime → Change runtime type → T4 GPU')
    raise SystemExit

print('\n📦 Installing dependencies...')

# Install packages quietly
import subprocess
import sys

packages = [
    'torch', 'torchvision', 'diffusers', 'transformers',
    'accelerate', 'imageio', 'imageio-ffmpeg',
    'opencv-python', 'pillow', 'trimesh', 'rembg'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('\n✅ All dependencies installed!')

In [ ]:
# @title 📤 Upload Your VibeMV Timeline
from google.colab import files
import json

print('📁 Upload your vibemv_timeline.json...')
uploaded = files.upload()

timeline_file = list(uploaded.keys())[0]
with open(timeline_file, 'r') as f:
    timeline = json.load(f)

print(f"\n✅ Loaded {len(timeline['scenes'])} scenes:")
for i, scene in enumerate(timeline['scenes']):
    print(f"  {i+1}. {scene['prompt'][:60]}...")

In [ ]:
# @title 🎨 Generate Images (SDXL on GPU)
from diffusers import StableDiffusionXLPipeline
import os

os.makedirs('generated_images', exist_ok=True)

print('Loading Stable Diffusion XL...')
pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16,
    variant='fp16'
).to('cuda')

scene_images = []
print(f"\n🎨 Generating {len(timeline['scenes'])} images...\n")

for i, scene in enumerate(timeline['scenes']):
    print(f"Scene {i+1}/{len(timeline['scenes'])}: {scene['prompt'][:50]}...")
    
    image = pipe(
        prompt=scene['prompt'],
        num_inference_steps=30,
        height=512,
        width=512
    ).images[0]
    
    img_path = f"generated_images/scene_{i:03d}.png"
    image.save(img_path)
    scene_images.append(img_path)
    print(f"  ✅ Saved")

del pipe
torch.cuda.empty_cache()
print(f"\n✅ Generated {len(scene_images)} images!")

In [ ]:
# @title 🎥 Create Video with Camera Motion
import cv2
import numpy as np
import imageio

def apply_camera_motion(img, motion, progress):
    h, w = img.shape[:2]
    
    if motion == 'zoom_in':
        scale = 1.0 + (0.3 * progress)
        new_h, new_w = int(h * scale), int(w * scale)
        zoomed = cv2.resize(img, (new_w, new_h))
        y1, x1 = (new_h - h) // 2, (new_w - w) // 2
        return zoomed[y1:y1+h, x1:x1+w]
    
    elif motion == 'orbit':
        scale = 1.0 + (0.2 * np.sin(progress * np.pi))
        new_h, new_w = int(h * scale), int(w * scale)
        zoomed = cv2.resize(img, (new_w, new_h))
        y1, x1 = (new_h - h) // 2, (new_w - w) // 2
        return zoomed[y1:y1+h, x1:x1+w]
    
    return img

fps = 24
all_frames = []

print('🎥 Generating video frames...\n')

for i, (scene, img_path) in enumerate(zip(timeline['scenes'], scene_images)):
    print(f"Processing scene {i+1}/{len(timeline['scenes'])}...")
    
    img = cv2.imread(img_path)
    duration = scene['duration']
    camera = scene.get('camera', 'static')
    num_frames = int(duration * fps)
    
    for frame_idx in range(num_frames):
        progress = frame_idx / max(num_frames - 1, 1)
        frame = apply_camera_motion(img.copy(), camera, progress)
        all_frames.append(frame)

print(f"\n✅ Generated {len(all_frames)} frames")

# Export video
print('\n💾 Creating video file...')
rgb_frames = [cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in all_frames]
imageio.mimsave('vibemv_video.mp4', rgb_frames, fps=fps, quality=8)

print('\n✅ Video created: vibemv_video.mp4')
files.download('vibemv_video.mp4')

---

## 🎲 Optional: 3D Model Generation

Run the cell below if you want to generate 3D models from your images.

**Note:** This adds ~1-2 minutes per scene.

In [ ]:
# @title 🎲 Generate 3D Models (Optional - Experimental)

print('⚠️  3D generation is experimental')
print('This will take ~60 seconds per scene')
print('\nSkip this if you only need video!\n')

generate_3d = input('Generate 3D models? (yes/no): ').lower()

if generate_3d == 'yes':
    print('\n📦 Installing TripoSR...')
    # Note: Full TripoSR integration requires more setup
    # For now, this is a placeholder
    print('⚠️  Full 3D generation coming soon!')
    print('For now, use images in Blender manually')
else:
    print('\n✅ Skipping 3D generation')

---

## ✅ All Done!

Your video has been downloaded!

### What You Got

- 🎥 **Video**: vibemv_video.mp4
- 🎨 **Images**: In `generated_images/` folder

### Next Steps

1. Check your downloads folder for the video
2. Upload back to VibeMV if needed
3. Or use directly!